# Instructions
Before you start this lesson you need to set up Google Colab a bit more:

1. Click the Runtime > Change runtime type menu option at the top of the page:
   ![Screenshot of change runtime type menu](https://github.com/catvec/machine-learning-camp/blob/main/assets/image-classifier/colab-change-runtime-type-menu.png?raw=true)
2. Then select "T4 GPU" under the hardware accelerator section and click "Save"
   ![Change runtime type popup screenshot](https://github.com/catvec/machine-learning-camp/blob/main/assets/image-classifier/colab-change-runtime-type-popup.png?raw=true)
3. Finally Click the "Run all" button at the top of the page:
   ![Screenshot showing run all button](https://github.com/catvec/machine-learning-camp/blob/main/assets/image-classifier/run-all-button.png?raw=true)
4. If you completed these instructions correctly you should see this message a few inches below on this page:
   ![Screenshot of image showing pytorch setup was successful](https://github.com/catvec/machine-learning-camp/blob/main/assets/image-classifier/colab-pytorch-accel-found.png?raw=true)
   If you don't see this message please verify you completed steps 1 and 2 correctly

In [ ]:
from collections import defaultdict

from ipywidgets import widgets
from IPython.display import display
import numpy as np
import matplotlib.pyplot as plt
import torch
from torch import nn
from torch.utils.data import random_split, Subset, Dataset, DataLoader
from torchvision import datasets, transforms, models

In [ ]:
# @title
if torch.accelerator.is_available():
    print("Excellent, you followed the setup steps correctly")
else:
    print("Whoops! It looks like Google Colab isn't setup correctly, please review the setup instructions above")

# Image Classifiers
In this notebook we are going to create a neural network which is trained to recognize objects in images. You will upload photos to use as training data to help the neural network learn what these objects look like.

The neural network we are going to use is a convolutional neural network.

## Where Will We Run the Neural Network?
Our plan is to run this neural network on a Raspberry Pi. This Raspberry Pi is a very small computer. Even smaller than your cellphone, and even less powerful.

Because we are running a neural network on this device we need to be careful about the structure of our neural network.

- Not only do we need to make a neural network which is effective
- It also needs to be efficient enough that we can run it on a small computer

Luckily the machine learning community has spent a lot of time figuring out how to do this. Researchers have created several neural networks focused on image classification on mobile devices.

- We will be using a neural network named MobileNet 2
- It is meant to run on smartphones
- Designed to recognize what object is in an image



## Training MobileNet 2
MobileNet 2 is a pretty big neural network:

- 53 layers
- 3.4 million parameters
- ~5,000 pixels of image as input
- Has to determine what object is in those pixels

To train a model like this you need a large dataset:

- 1.28 million images
- Of 1,000 objects

Training a big model like this with such a large dataset takes days:

- With 8 large expensive GPUs
- Running non-stop
- Training would take 2-3 days

Because of this we run in to several problems:

- We don't have 8 large expensive GPUs
- We don't have multiple days to train our neural network

### Fine Tuning
There is a solution to our problem of training MobileNet 2. We can do a process called **fine tuning**.

#### What Does Training MobileNet 2 Do?
During the process of training MobileNet 2 what is the neural network learning?

- Sure it's learning how to identify the 1,000 objects in the training dataset
- But what else?
- It's learning how to differentiate between objects
- It learns to look at how sharp corners are
- Or to pay attention to outlines

During training MobileNet 2 is learning general skills that can be used to recognize any object.

- It's like how in math class you learn how to do math by practicing solving equations
- After a year of math you aren't just able to solve the equations you saw on your worksheets and tests
- You've built the skills required to solve similar equations

The same is true of a complex neural network like MobileNet 2. After training it's built up the skills required to recognize and differentiate between objects in general.

#### Fine Tuning MobileNet 2
We can take advantage of the fact that after training MobileNet 2 has the skills to look at and understand objects.

- Instead of starting our training process with random weights and biases
- We start the training process with the weights and biases from the fully trained MobileNet 2 model
  - This gives us a 2-3 day headstart on training
  - Now right off the bat our neural network has good enoughts weights and biases to correctly categorize 1,000 objects

We take the output layer of MobileNet 2 and replace it with our own.

- Instead of 1,000 output neurons
  - 1 for each object the model can recognize
  - Each outputing a number from 0.0 to 1.0
  - A percentage from 0% to 100% of how sure the model is that the image is of that object
- Our replacement output layer will have the number of neurons equal to the number of objects we want to recognize
  - So if we only want to categorize dogs versus cats
  - We have 2 output neurons
  - Each outputting 0.0 to 1.0 as a probability of how sure the model is that a dog or cat is in the image

Then during training we don't touch the weights or biases for any other layer. We only modify the weights and biases of our final output layer.

- This lets MobileNet 2 retain all of its skills that it built up for recognizing objects
- By only training our final layer we need a much smaller computer and much less time to train
- We don't have millions of photos of the objects we want to categorize

In reality, few people ever train a complex neural network from scratch. For the exact reasons we mentioned above. A lot of the time people using the process of fine tuning to repurpose existing models.

# Check In - MobileNet 2 Fine Tuning
We just learned about MobileNet 2 and the process of fine tuning. Here are the points you should understand:

- MobileNet 2 is a neural network made to run on not so powerful devices like cell phones
- It would take way too long and cost way too much money to train MobileNet 2 from scratch
- Fine tuning is the process of replacing the output layer of an existing neural network
- Fine tuning let's us repurpose an already trained neural network for our specific task

# Training Data
Since we are fine tuning our mo

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
datasets_parent = "/content/drive/MyDrive/Work/Putny Nat Geo/Putney as Collab/Datasets"
dataset = datasets.ImageFolder(datasets_parent)

In [ ]:
mobilenet_mean = [0.485, 0.456, 0.406]
mobilenet_std = [0.229, 0.224, 0.225]

training_transforms = transforms.Compose([
    transforms.RandomResizedCrop(224),
    transforms.RandomHorizontalFlip(),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=mobilenet_mean,
        std=mobilenet_std,
    )
])
validation_transforms = transforms.Compose([
    transforms.Resize(256),
    transforms.CenterCrop(224),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=mobilenet_mean,
        std=mobilenet_std,
    )
])

unnormalize_transform = transforms.Normalize(
    mean=[-m / s for m, s in zip(mobilenet_mean, mobilenet_std)],
    std=[1 / s for s in mobilenet_std]
)

In [ ]:
class TransformDataset(Dataset):
    def __init__(self, dataset, transform=None):
        self.dataset = dataset
        self.transform = transform

    def __len__(self):
        return len(self.dataset)

    def __getitem__(self, idx):
        sample, label = self.dataset[idx]
        sample = self.transform(sample)
        return sample, label

In [ ]:
# For each image class get the dataset item indexes
class_idxs = defaultdict(list)
for sample_idx, label in enumerate([ sample[1] for sample in dataset.samples ]):
    class_idxs[label].append(sample_idx)

# Shuffle the indexes of each item for each class
for label, indexes in class_idxs.items():
    class_idxs[label] = np.random.permutation(indexes).tolist()

# Create dataset subsets
training_pct = 0.6
test_pct = 0.2
eval_pct = 0.2

training_idxs = []
test_idxs = []
eval_idxs = []

for label, indexes in class_idxs.items():
    training_size = int(len(indexes) * training_pct)
    test_size = int(len(indexes) * test_pct)
    eval_size = len(indexes) - (training_size + test_size)

    training_idxs.extend(indexes[:training_size])
    test_idxs.extend(indexes[training_size:training_size + test_size])
    eval_idxs.extend(indexes[training_size + test_size:])

training_dataset = Subset(TransformDataset(dataset, training_transforms), training_idxs)
test_dataset = Subset(TransformDataset(dataset, validation_transforms), test_idxs)
eval_dataset = Subset(TransformDataset(dataset, validation_transforms), eval_idxs)

In [ ]:
training_loader = DataLoader(training_dataset, batch_size=4, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=4, shuffle=False)
eval_loader = DataLoader(eval_dataset, batch_size=4, shuffle=False)

In [ ]:
def imshow(inp, title = None):
    # Un-normalize image pixel data
    inp = unnormalize_transform(inp)

    # Flip pixel channels into correct value for displaying
    inp = inp.numpy().transpose((1, 2, 0))

    # Ensure values aren't too big or small to display
    inp = np.clip(inp, 0, 1)

    # Show image
    display(transforms.ToPILImage()(inp))

    # Show title
    if title is not None:
        display(title)

num_shown = 0
for batch in training_loader:
    for img, label in zip(*batch):
       imshow(img, dataset.classes[label])
       num_shown += 1

       if num_shown > 20:
          break
    if num_shown > 20:
          break

In [ ]:
model = models.mobilenet_v2(pretrained=True)

In [ ]:
model.classifier[1] = nn.Linear(model.classifier[1].in_features, len(dataset.classes))

In [ ]:
device = torch.accelerator.current_accelerator().type if torch.accelerator.is_available() else "cpu"
print(f"Using {device} device")

# Setup the neural network
model = model.to(device)

# Define how we're going to train the model
learning_rate = 0.0005
num_training_rounds = 50

error_fn = nn.CrossEntropyLoss()
optimizer = torch.optim.SGD(model.classifier[1].parameters(), lr=0.001, momentum=0.9)

stop_training_below_error = 0.05

def training_loop(model, loader, optimizer, error_fn):
    # Function which runs our model through a dataset and returns the error
    total_error = 0.0

    for batch_input, batch_labels in loader:
        # See what the model predicts
        prediction = model(batch_input.to(device))

        # Calculate the error (how close the prediction was to the correct result)
        error = error_fn(prediction, batch_labels.to(device))

        total_error = total_error + error

        # If an optimizer is specified then updates the model parameters using it
        if optimizer is not None:
            # Get the derivative of all the weights and biases in relation to the error equation
            error.backward()

            # Update the weights and biases by a small amount in the correct direction
            optimizer.step()

            # Reset the derivatives to zero so we can re-calculate them for the next dataset item
            optimizer.zero_grad()

    return total_error


# Training loop
for training_round in range(num_training_rounds):
    # Update the parameters of our model with the training dataset
    # Tell our model it's about to be trained
    model.train()

    training_loop(
        model=model,
        loader=training_loader,
        optimizer=optimizer,
        error_fn=error_fn,
      )

    # Use the test dataset to determine how training is going
    # Tell our model it's about to be evaluated
    model.eval()

    test_error = training_loop(
        model=model,
        loader=test_loader,
        optimizer=None, # Do not provide an optimizer, so parameters are not updated
        error_fn=error_fn,
      )

    # Stop training if our error is low enough
    if test_error <= stop_training_below_error:
        print(f"In training round {training_round} we achieved test error {test_error}, exiting early")
        break

    # Print our progress every training round
    print(f"Training round {training_round}, test error {test_error}")

# Finally see how accurate our model is via the evaluation dataset
# Tell our model it's about to be evaluated
model.eval()

eval_error = training_loop(
    model=model,
    loader=eval_loader,
    optimizer=None, # Do not provide an optimizer, so parameters are not updated
    error_fn=error_fn,
)

print(f"After {training_round+1} training rounds the model has an evaluation error of {eval_error}")

In [ ]:
correct = 0
wrong = 0

for batch in eval_loader:
    for img, class_ in zip(*batch):
        prediction = model(img.unsqueeze(0).to(device))

        predicted_class = prediction.argmax().item()
        predicted_label = dataset.classes[predicted_class]

        actual_label = dataset.classes[class_]

        if predicted_class != class_:
            wrong += 1
        else:
            correct += 1

        title = f"Predicted: {predicted_label} ({prediction.tolist()[0][predicted_class]}), Actual: {actual_label}"
        imshow(img, title)
        num_shown += 1

print(f"correct={correct}, wrong={wrong}")